# 00 · Manual de `src`

Notebook demostrativo que recorre los módulos de `src/` usando datos sintéticos de **scikit-learn**.  
No requiere datos reales: ideal para verificar que el entorno y la estructura del proyecto funcionan correctamente.

| Sección | Módulo |
|---|---|
| 1. Paths | `src.utils.paths` |
| 2. Utils | `src.utils.general_functions.Utils` |
| 3. Features | `src.features.build_features` |
| 4. Models | `src.models.models.Models` |
| 5. Visualize | `src.visualization.visualize` |

In [ ]:
import sys
from pyprojroot import here

# Ensure src/ is importable regardless of how the notebook is launched
sys.path.insert(0, str(here()))
print('Project root:', here())

## 1 · Paths

`src/utils/paths.py` expone funciones de conveniencia para construir rutas
relativas a la raíz del proyecto usando `pyprojroot.here()`.

In [ ]:
from src.utils.paths import (
    project_dir,
    data_raw_dir,
    data_processed_dir,
    models_dir,
    reports_figures_dir,
)

print('project_dir()       :', project_dir())
print('data_raw_dir()      :', data_raw_dir())
print('data_processed_dir():', data_processed_dir())
print('models_dir()        :', models_dir())
print('reports_figures_dir():', reports_figures_dir())

# Paths accept extra segments
print('models_dir("run1")  :', models_dir('run1'))

## 2 · Utils

`Utils` centraliza las operaciones comunes de ML: carga, split, escalado y exportación de modelos.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_regression

from src.utils.general_functions import Utils

# ── Generate synthetic regression dataset ──────────────────────────────────
X_np, y_np = make_regression(
    n_samples=300, n_features=8, noise=0.1, random_state=42
)
feature_names = [f'feat_{i}' for i in range(X_np.shape[1])]
df = pd.DataFrame(X_np, columns=feature_names)
df['target'] = y_np

print('Shape:', df.shape)
df.head(3)

In [ ]:
utils = Utils()

# features_target ───────────────────────────────────────────────────────────
X, y = utils.features_target(df, drop_cols=['target'], y='target')
print('X shape:', X.shape, '| y shape:', y.shape)

# split_data ────────────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = utils.split_data(X, y, size=0.2)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

# process_and_scale_data ───────────────────────────────────────────────────
X_train_sc, X_test_sc = utils.process_and_scale_data(
    X_train, X_test, scaler_name='demo_scaler.joblib'
)
print('Scaled train mean (should ≈ 0):', np.mean(X_train_sc).round(4))
print('Scaled train std  (should ≈ 1):', np.std(X_train_sc).round(4))

## 3 · Build Features

`build_features.py` incluye `describe_dataset` para estadísticas descriptivas
y `check_normality` para evaluar distribuciones.

In [ ]:
from src.features.build_features import describe_dataset

summary = describe_dataset(df)
print('Columnas cuantitativas analizadas:', list(summary['quantitative'].index))
summary['quantitative'][['mean', 'std_dev', 'skewness', 'kurtosis']].head()

In [ ]:
from src.features.build_features import check_normality

# Use only 2 features to keep the visual output compact
normality_results = check_normality(df[['feat_0', 'feat_1']])
normality_results

## 4 · Models

`Models.grid_training` realiza búsqueda de hiperparámetros con `GridSearchCV`
sobre SVR y GradientBoostingRegressor, y exporta el mejor modelo a `models/`.

In [ ]:
from src.models.models import Models

# Use a small subset so grid search is fast in demo mode
models = Models()

# Narrow the grid for speed
models.param['SVR'] = {'C': [1, 5], 'gamma': ['scale'], 'kernel': ['rbf']}
models.param['GRADIENT'] = {'loss': ['squared_error'], 'learning_rate': [0.05, 0.1]}

models.grid_training(X_train_sc, y_train)
print('\nBest model saved to:', models_dir())

## 5 · Visualize

`visualize.py` ofrece helpers listos para reportes: importancia de features y análisis de residuos.

In [ ]:
import joblib
from sklearn.ensemble import RandomForestRegressor
from src.visualization.visualize import plot_feature_importance, plot_residuals

# Train a quick RandomForest to get feature importances (interpretable)
rf = RandomForestRegressor(n_estimators=50, random_state=42)
rf.fit(X_train_sc, y_train)

fig_importance = plot_feature_importance(
    feature_names=feature_names,
    importances=rf.feature_importances_,
    title='Feature Importance — Synthetic Data',
    save=False,
)
fig_importance.show()

In [ ]:
y_pred = rf.predict(X_test_sc)

fig_residuals = plot_residuals(
    y_true=y_test,
    y_pred=y_pred,
    title='Residuals — RandomForest on Synthetic Data',
    save=False,
)
fig_residuals.show()

---

✅ **All `src/` modules verified with synthetic data.**  
Replace the `make_regression` block in Section 2 with `utils.load_dataset(<path>)` to work with real data.